In [1]:
import os
import cv2
import joblib
import pandas as pd
import numpy as np
from mediapipe.tasks.python import vision

# Safe imports from your existing codebase
import import_ipynb
from main import (
    options, 
    extract_coordinates_from_video, 
    add_smoothed_features, 
    add_repetitions_column, 
    add_velocity_from_smoothed, 
    add_relative_position_features,
    build_single_rep_vector
)
from ml import COLUMNS_TO_DROP  # Standard tracking filters to keep input matrix aligned

def batch_process_and_evaluate(input_folder_path="../user_videos/", model_dir="../models/"):
    # 1. Load trained ML pipeline assets
    model_path = os.path.join(model_dir, "squat_classifier_model.pkl")
    scaler_path = os.path.join(model_dir, "squat_scaler.pkl")
    
    if not os.path.exists(model_path) or not os.path.exists(scaler_path):
        raise FileNotFoundError("Missing machine learning models in storage. Please complete model training first.")
        
    model = joblib.load(model_path)
    scaler = joblib.load(scaler_path)
    
    # 2. Identify video files inside target directory
    valid_exts = (".mp4", ".avi", ".mov", ".mkv")
    if not os.path.exists(input_folder_path):
        print(f"Creating missing tracking folder: {input_folder_path}")
        os.makedirs(input_folder_path, exist_ok=True)
        return
        
    video_files = [f for f in os.listdir(input_folder_path) if f.lower().endswith(valid_exts)]
    if not video_files:
        print(f"No execution targets. Place squat clips inside '{input_folder_path}'.")
        return

    print(f"🚀 Found {len(video_files)} target videos. Starting biomechanical pipeline evaluation...\n")

    for video_file in video_files:
        video_path = os.path.join(input_folder_path, video_file)
        print(f"Processing: {video_file}")
        
        # Identify video FPS
        cap = cv2.VideoCapture(video_path)
        fps = cap.get(cv2.CAP_PROP_FPS) or 30.0
        cap.release()
        
        # FIX: Open a FRESH landmarker context instance for each individual video file.
        # This completely resets MediaPipe's inner timestamp clock tracking state back to zero.
        with vision.PoseLandmarker.create_from_options(options) as landmarker:
            raw_rows = extract_coordinates_from_video(video_path, label="inference", landmarker=landmarker)
            
        if not raw_rows:
            print(f"❌ Skipping {video_file}: MediaPipe failed to lock on subject joints.")
            continue
            
        df = pd.DataFrame(raw_rows)
        
        # Step B: Run data science transformations in identical sequence
        df = add_smoothed_features(df)
        df = add_repetitions_column(df)
        df = add_velocity_from_smoothed(df)
        df = add_relative_position_features(df)
        
        # Isolate valid rows that successfully completed a full repetition trajectory
        rep_df_groups = df[df['rep_number'] > 0]
        if rep_df_groups.empty:
            print(f"⚠️  No valid repetitions segmented for {video_file}. Verify lift depth.")
            continue
            
        # Step C: Establish Fatigue Performance baseline context using Repetition 1
        rep1_frames = rep_df_groups[rep_df_groups['rep_number'] == 1]
        if not rep1_frames.empty:
            bottom_idx_rep1 = rep1_frames['hip_y_smooth'].idxmax()
            ascent_rep1 = rep1_frames.loc[bottom_idx_rep1:]
            rep_1_avg_ascent_vel = abs(ascent_rep1['hip_velocity_y'].mean()) if not ascent_rep1.empty else None
        else:
            rep_1_avg_ascent_vel = None

        print(f"📋 Found {rep_df_groups['rep_number'].nunique()} reps inside {video_file}:")
        
        # Step D: Process structured sub-chunks
        for rep_num, rep_data in rep_df_groups.groupby('rep_number'):
            # Extract master vector using clean single function from main
            rep_vector_dict = build_single_rep_vector(
                rep_df=rep_data,
                video_name=video_file,
                rep_num=rep_num,
                rep_1_avg_ascent_vel=rep_1_avg_ascent_vel,
                fps=fps
            )
            
            # Format into single row dataframe to structure model inputs safely
            vector_df = pd.DataFrame([rep_vector_dict])
            features_to_scale = vector_df.drop(columns=[col for col in COLUMNS_TO_DROP if col in vector_df.columns])
            
            # Scale variables and perform prediction matching your ml script matrix
            scaled_features = scaler.transform(features_to_scale)
            prediction = model.predict(scaled_features)[0]
            
            # Form feedback output display
            status_icon = "✅ GOOD" if prediction == "good" else "❌ BAD FORM"
            print(f"   • Rep #{rep_num}: {status_icon}")
            print(f"     [Side: {rep_vector_dict['body_side']} | Lean: {rep_vector_dict.get('max_torso_lean_degrees', 0):.1f}° | Ascent Vel: {rep_vector_dict.get('peak_ascent_velocity_y', 0):.4f}]")
        print("-" * 50)


'../edited_datasets\squat_features.csv' already exists. Skipping dataset creation.
Saved engineered features to '../edited_datasets/squat_features_engineered.csv'
Shape: (8308, 33)

--- Dataset Assembly Complete ---
Successfully processed 35 total individual repetitions.
Final Data Matrix Dimensions: (35, 35)
File stored safely at: '../edited_datasets/squat_rep_vectors_ml.csv'
      visibility                    video_name
4613    0.909228  good_no_weights_personal.mp4
4614    0.909237  good_no_weights_personal.mp4
4616    0.909318  good_no_weights_personal.mp4
4617    0.909605  good_no_weights_personal.mp4
4615    0.909863  good_no_weights_personal.mp4
4612    0.910594  good_no_weights_personal.mp4
4610    0.911458  good_no_weights_personal.mp4
4618    0.911580  good_no_weights_personal.mp4
4625    0.911680  good_no_weights_personal.mp4
4624    0.911726  good_no_weights_personal.mp4
Dataset Loaded: 35 repetitions, 18 features.

--- Starting Leak-Proof Cross Validation ---
Fold 1: 1.00

c:\Users\Anton\Desktop\ExerciseTechnique\.venv\Lib\site-packages\sklearn\svm\_base.py:239: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(
c:\Users\Anton\Desktop\ExerciseTechnique\.venv\Lib\site-packages\sklearn\svm\_base.py:239: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(
c:\Users\Anton\Desktop\ExerciseTechnique\.venv\Lib\site-packages\sklearn\svm\_base.py:239: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(
c:\Users\Anton\Desktop\ExerciseTechnique\.venv\Lib\site-packages\sklearn\svm\_base.py:239: FutureWarning: The

In [2]:
batch_process_and_evaluate()

🚀 Found 2 target videos. Starting biomechanical pipeline evaluation...

Processing: left_girl.mp4
Processing: left_girl.mp4...
📋 Found 2 reps inside left_girl.mp4:
   • Rep #1: ❌ BAD FORM
     [Side: right | Lean: 24.0° | Ascent Vel: 0.0088]
   • Rep #2: ❌ BAD FORM
     [Side: right | Lean: 33.7° | Ascent Vel: 0.0125]
--------------------------------------------------
Processing: right_girl.mp4
Processing: right_girl.mp4...
📋 Found 2 reps inside right_girl.mp4:
   • Rep #1: ❌ BAD FORM
     [Side: left | Lean: 41.0° | Ascent Vel: 0.0087]
   • Rep #2: ❌ BAD FORM
     [Side: left | Lean: 45.3° | Ascent Vel: 0.0074]
--------------------------------------------------
